# Fusion Random Forest — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `fused_landslide_training_dataset.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn joblib

import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_FILE = "fused_landslide_training_dataset.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)

FEATURES = [
    "pressure", "soil_moisture",
    "water_level_cm", "rain_intensity_percent", "water_sensor_percent",
    "flood_detected", "flood_risk_num",
    "motion", "vibration_detected", "earthquake_detected", "eq_risk_num",
    "water_motion_interaction", "rain_soil_interaction"
]
TARGET = "land_risk_num"
CLASS_NAMES = ["Low", "Medium", "High", "Critical"]

df2 = df.dropna(subset=FEATURES + [TARGET]).copy()
X = df2[FEATURES].astype(float).to_numpy()
y = df2[TARGET].astype(int).to_numpy()

print("Rows:", len(df2))
print("Class counts:", pd.Series(y).value_counts().sort_index().to_dict())

## Leakage-free fusion evaluation with validation-only threshold tuning

The test set is isolated first and never used to choose the Critical-class threshold.

1. Hold out 20% test data with stratification.
2. Split the remaining 80% into training and validation.
3. Fit scaler + RF on training only.
4. Select the Critical probability threshold using **validation only**.
5. Freeze the threshold.
6. Refit the final scaler + RF on the entire 80% development set.
7. Evaluate once on the untouched 20% test set.

In [ ]:
X_dev, X_test_raw, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.20,
    random_state=SEED,
    stratify=y_dev
)

scaler_tune = MinMaxScaler()
X_train = scaler_tune.fit_transform(X_train_raw)
X_val   = scaler_tune.transform(X_val_raw)

tune_model = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1
)
tune_model.fit(X_train, y_train)

val_proba = tune_model.predict_proba(X_val)
val_argmax = np.argmax(val_proba, axis=1)

print("Validation default macro-F1:", f1_score(y_val, val_argmax, average="macro", zero_division=0))

In [ ]:
candidates = np.round(np.arange(0.10, 0.51, 0.01), 2)
records = []

for threshold in candidates:
    pred = np.argmax(val_proba, axis=1)
    pred = np.where(val_proba[:, 3] >= threshold, 3, pred)

    macro_f1 = f1_score(y_val, pred, average="macro", zero_division=0)
    crit_recall = recall_score(
        (y_val == 3).astype(int),
        (pred == 3).astype(int),
        zero_division=0
    )
    records.append((threshold, macro_f1, crit_recall))

threshold_table = pd.DataFrame(records, columns=["threshold", "macro_f1", "critical_recall"])
best_row = threshold_table.sort_values(
    ["macro_f1", "critical_recall"],
    ascending=[False, False]
).iloc[0]

BEST_THRESHOLD = float(best_row["threshold"])
print("Selected validation-only Critical threshold:", BEST_THRESHOLD)
print(best_row)

In [ ]:
final_scaler = MinMaxScaler()
X_dev_scaled = final_scaler.fit_transform(X_dev)
X_test = final_scaler.transform(X_test_raw)

final_model = RandomForestClassifier(
    n_estimators=500,
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1
)
final_model.fit(X_dev_scaled, y_dev)

test_proba = final_model.predict_proba(X_test)

pred_default = np.argmax(test_proba, axis=1)

pred_thresholded = np.argmax(test_proba, axis=1)
pred_thresholded = np.where(test_proba[:, 3] >= BEST_THRESHOLD, 3, pred_thresholded)

def report(name, pred):
    print("\n===", name, "===")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Macro F1:", f1_score(y_test, pred, average="macro", zero_division=0))
    print(classification_report(
        y_test, pred,
        labels=[0,1,2,3],
        target_names=CLASS_NAMES,
        zero_division=0
    ))
    cm = confusion_matrix(y_test, pred, labels=[0,1,2,3])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(values_format="d")
    plt.title(name)
    plt.show()

report("Fusion RF — Default Decision", pred_default)
report(f"Fusion RF — Validation-Tuned Critical Threshold ({BEST_THRESHOLD:.2f})", pred_thresholded)

pd.DataFrame({
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "pred_default": [CLASS_NAMES[i] for i in pred_default],
    "pred_thresholded": [CLASS_NAMES[i] for i in pred_thresholded]
}).to_csv("fusion_rf_leakage_free_test_predictions.csv", index=False)

In [ ]:
tree_depths = np.array([est.tree_.max_depth for est in final_model.estimators_])
print("Number of trees:", len(final_model.estimators_))
print("Mean tree depth :", tree_depths.mean())
print("Max tree depth  :", tree_depths.max())

In [ ]:
bundle = {
    "features": FEATURES,
    "class_names": CLASS_NAMES,
    "scaler": final_scaler,
    "model": final_model,
    "critical_threshold": BEST_THRESHOLD,
    "test_policy": "threshold selected on validation only; final test untouched until evaluation",
    "seed": SEED
}
joblib.dump(bundle, "fusion_rf_leakage_free_bundle.joblib")
threshold_table.to_csv("fusion_validation_threshold_search.csv", index=False)

print("Saved fusion RF bundle and validation threshold search table.")